<a href="https://colab.research.google.com/github/lekhasree15/BIS_LAB/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

import random

# ---------------- TASK DETAILS ----------------

tasks = [
    {"id": 0, "cpu": 2, "ram": 4, "priority": 3, "deadline": 12, "deps": []},
    {"id": 1, "cpu": 3, "ram": 2, "priority": 2, "deadline": 18, "deps": [0]},
    {"id": 2, "cpu": 4, "ram": 4, "priority": 3, "deadline": 25, "deps": [0]},
    {"id": 3, "cpu": 2, "ram": 2, "priority": 1, "deadline": 30, "deps": [1]},
    {"id": 4, "cpu": 5, "ram": 4, "priority": 2, "deadline": 35, "deps": [1, 2]},
    {"id": 5, "cpu": 3, "ram": 3, "priority": 1, "deadline": 40, "deps": [3]},
    {"id": 6, "cpu": 4, "ram": 2, "priority": 3, "deadline": 45, "deps": [4]},
    {"id": 7, "cpu": 2, "ram": 3, "priority": 2, "deadline": 50, "deps": [5, 6]}
]

# ---------------- VIRTUAL MACHINES ----------------

vms = [
    {"speed": 2, "ram": 8, "power": 5},
    {"speed": 3, "ram": 6, "power": 7},
    {"speed": 4, "ram": 8, "power": 10},
    {"speed": 5, "ram": 12, "power": 13}
]

NUM_TASKS = len(tasks)
NUM_VMS = len(vms)

# ---------------- FITNESS EVALUATION ----------------

def evaluate(schedule):

    vm_available = [0] * NUM_VMS
    finish_time = {}
    total_energy = 0
    sla_violations = 0
    resource_penalty = 0

    for task in tasks:

        task_id = task["id"]
        vm_id = schedule[task_id]

        vm = vms[vm_id]

        # Check VM memory capacity
        if task["ram"] > vm["ram"]:
            resource_penalty += 100

        # Calculate task execution time
        duration = task["cpu"] / vm["speed"]

        # Find dependency completion time
        dependency_time = 0

        for dep in task["deps"]:
            dependency_time = max(
                dependency_time, finish_time[dep]
            )

        # Task starts after dependencies and VM become available
        start_time = max(
            dependency_time, vm_available[vm_id]
        )

        end_time = start_time + duration

        # Update VM availability
        vm_available[vm_id] = end_time

        # Store completion time
        finish_time[task_id] = end_time

        # Calculate energy consumption
        total_energy += duration * vm["power"]

        # Check SLA deadline
        if end_time > task["deadline"]:
            sla_violations += 1

    makespan = max(finish_time.values())

    # Combined objective: lower is better
    fitness = (
        makespan
        + 0.1 * total_energy
        + 20 * sla_violations
        + resource_penalty
    )

    return fitness, makespan, total_energy, sla_violations


# ---------------- CREATE INITIAL NESTS ----------------

def create_nest():

    schedule = []

    for task in tasks:

        possible_vms = []

        for j in range(NUM_VMS):
            if task["ram"] <= vms[j]["ram"]:
                possible_vms.append(j)

        schedule.append(random.choice(possible_vms))

    return schedule


# ---------------- LEVY FLIGHT ----------------

def levy_flight(schedule):

    new_schedule = schedule.copy()

    # Generate a random number of task changes
    changes = min(
        NUM_TASKS,
        max(1, int(random.paretovariate(1.5)))
    )

    for i in range(changes):

        task_id = random.randint(0, NUM_TASKS - 1)

        possible_vms = []

        for j in range(NUM_VMS):
            if tasks[task_id]["ram"] <= vms[j]["ram"]:
                possible_vms.append(j)

        new_schedule[task_id] = random.choice(possible_vms)

    return new_schedule


# ---------------- CROSSOVER ----------------

def crossover(parent1, parent2):

    child = []

    for i in range(NUM_TASKS):

        if random.random() < 0.5:
            child.append(parent1[i])
        else:
            child.append(parent2[i])

    return child


# ---------------- MUTATION ----------------

def mutation(schedule):

    new_schedule = schedule.copy()

    if random.random() < 0.3:

        task_id = random.randint(0, NUM_TASKS - 1)

        possible_vms = []

        for j in range(NUM_VMS):
            if tasks[task_id]["ram"] <= vms[j]["ram"]:
                possible_vms.append(j)

        new_schedule[task_id] = random.choice(possible_vms)

    return new_schedule


# ---------------- CUCKOO SEARCH ALGORITHM ----------------

def cuckoo_search():

    population_size = 15
    generations = 100
    discovery_probability = 0.25

    # Create initial nests
    nests = []

    for i in range(population_size):
        nests.append(create_nest())

    best_nest = None
    best_fitness = float("inf")

    # Find initial best solution
    for nest in nests:

        fitness, _, _, _ = evaluate(nest)

        if fitness < best_fitness:
            best_fitness = fitness
            best_nest = nest.copy()

    # Main optimization loop
    for generation in range(generations):

        # Generate new solutions using Levy Flight
        for i in range(population_size):

            new_nest = levy_flight(nests[i])

            new_fitness, _, _, _ = evaluate(new_nest)

            random_index = random.randint(0, population_size - 1)

            old_fitness, _, _, _ = evaluate(nests[random_index])

            if new_fitness < old_fitness:
                nests[random_index] = new_nest.copy()

        # Apply crossover and mutation
        for i in range(population_size):

            parent1 = nests[i]
            parent2 = random.choice(nests)

            child = crossover(parent1, parent2)
            child = mutation(child)

            child_fitness, _, _, _ = evaluate(child)
            current_fitness, _, _, _ = evaluate(nests[i])

            if child_fitness < current_fitness:
                nests[i] = child.copy()

        # Abandon some poor nests
        scores = []

        for nest in nests:
            fitness, _, _, _ = evaluate(nest)
            scores.append(fitness)

        worst_indices = sorted(
            range(population_size),
            key=lambda x: scores[x],
            reverse=True
        )

        number_to_replace = int(
            population_size * discovery_probability
        )

        for i in range(number_to_replace):
            index = worst_indices[i]
            nests[index] = create_nest()

        # Update best solution
        for nest in nests:

            fitness, _, _, _ = evaluate(nest)

            if fitness < best_fitness:
                best_fitness = fitness
                best_nest = nest.copy()

    return best_nest


# ---------------- MAIN PROGRAM ----------------

best_schedule = cuckoo_search()

fitness, makespan, energy, sla = evaluate(best_schedule)

print("\n===== CUCKOO SEARCH CLOUD TASK SCHEDULING =====")

print("\nOptimal Task Allocation:")

for i in range(NUM_TASKS):
    print(
        "Task", i,
        "-> Virtual Machine", best_schedule[i]
    )

print("\nPerformance Results:")

print("Makespan:", round(makespan, 2))
print("Energy Consumption:", round(energy, 2))
print("SLA Violations:", sla)
print("Best Fitness Value:", round(fitness, 2))

print("\nScheduling Completed Successfully!")

#https://ieeexplore.ieee.org/document/8974273



===== CUCKOO SEARCH CLOUD TASK SCHEDULING =====

Optimal Task Allocation:
Task 0 -> Virtual Machine 3
Task 1 -> Virtual Machine 2
Task 2 -> Virtual Machine 3
Task 3 -> Virtual Machine 1
Task 4 -> Virtual Machine 3
Task 5 -> Virtual Machine 1
Task 6 -> Virtual Machine 3
Task 7 -> Virtual Machine 3

Performance Results:
Makespan: 3.4
Energy Consumption: 63.37
SLA Violations: 0
Best Fitness Value: 9.74

Scheduling Completed Successfully!
